# GreenMind AI: Notebook 1, build the iNaturalist dataset and test the deployed model

**Settings (right panel):** Accelerator = *None* (CPU is enough), **Internet = On**.
Then click **Save Version → Save & Run All (Commit)**. It runs in the background (about 3–5 h); you can close the browser.

What it does:
1. Downloads research-grade CC-licensed iNaturalist photos for the 1,106 species of the deployed model:
   - `pool_a`: observations uploaded **before 2025-09-01**, used for training
   - `pool_b`: observations uploaded **after 2025-09-03**, after the deployed model was saved, so it has never seen them
2. Evaluates the deployed ResNet-18 on `pool_b`.

When it finishes, open Notebook 2 and add this notebook's output as its input.

In [ ]:
import os, subprocess, sys
os.chdir("/kaggle/working")
if not os.path.exists("greenmind-ai"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/mubashir20023/greenmind-ai"], check=True)
os.chdir("/kaggle/working/greenmind-ai")
assert os.path.exists("tools/inat_fetch_species.py"), "Push the new tools/ folder to GitHub first (see RUN_EXPERIMENTS.md, Step 0)"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm==1.0.19", "scikit-learn"], check=False)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
# Get the deployed ResNet-18 checkpoint (stored with Git LFS)
import os, subprocess, urllib.request
ck = "models/checkpoints/plant_id_resnet18_best.pth"
if os.path.getsize(ck) < 10_000:
    r = subprocess.run("git lfs install --local && git lfs pull --include=" + ck, shell=True)
if os.path.getsize(ck) < 10_000:
    urllib.request.urlretrieve(
        "https://media.githubusercontent.com/media/mubashir20023/greenmind-ai/main/" + ck, ck)
print(ck, os.path.getsize(ck) // 1_000_000, "MB")
assert os.path.getsize(ck) > 40_000_000

In [ ]:
!python tools/inat_fetch_species.py --out /kaggle/working/inat --per_class_a 150 --per_class_b 20 --workers 16

In [ ]:
!python tools/evaluate.py --ckpt models/checkpoints/plant_id_resnet18_best.pth --arch resnet18 \
    --classes models/class_maps/idx_to_species.json --imagefolder /kaggle/working/inat/pool_b \
    --workers 4 --out /kaggle/working/results/e1_deployed_resnet18_temporal

In [ ]:
!cat /kaggle/working/inat/fetch_stats.json
!cat /kaggle/working/results/e1_deployed_resnet18_temporal/metrics.json